<a href="https://colab.research.google.com/github/jayanthkorupolu2000-web/Final_Project/blob/main/EfficientNetV2_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers timm


In [ ]:
import tensorflow as tf

# Load the fine-tuned hybrid model
loaded_model = tf.keras.models.load_model("/content/drive/MyDrive/hybrid_finetuned_model.keras")

print("Hybrid fine-tuned model loaded successfully!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random
import numpy as np
import tensorflow as tf
from PIL import Image
from collections import defaultdict


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_PATH = "/content/drive/MyDrive/crack_dataset"
TRAIN_IMG_DIR = BASE_PATH + "/train"

IMG_SIZE = 224
BATCH_SIZE = 16
MIN_CRACK_AREA = 50


In [ ]:
with open(BASE_PATH + "/train/_annotations.coco.json", "r") as f:
    coco = json.load(f)

img_id_to_name = {img["id"]: img["file_name"] for img in coco["images"]}

# group bboxes per image
img_to_bboxes = defaultdict(list)
for ann in coco["annotations"]:
    img_to_bboxes[ann["image_id"]].append(ann["bbox"])

print("Images:", len(img_to_bboxes))
print("Annotations:", len(coco["annotations"]))


Images: 735
Annotations: 4233


In [ ]:
image_ids = list(img_to_bboxes.keys())
random.shuffle(image_ids)

split_idx = int(0.8 * len(image_ids))
train_img_ids = set(image_ids[:split_idx])
val_img_ids   = set(image_ids[split_idx:])

print("Train images:", len(train_img_ids))
print("Val images  :", len(val_img_ids))


Train images: 588
Val images  : 147


In [ ]:
# Generates crack and background crops (image-level split, no leakage)
def train_generator():
    for img_id in train_img_ids:
        img_name = img_id_to_name[img_id]
        img_path = os.path.join(TRAIN_IMG_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        bboxes = img_to_bboxes[img_id]

        # positive samples
        for x, y, w, h in bboxes:
            x, y, w, h = map(int, [x, y, w, h])
            if w * h < MIN_CRACK_AREA:
                continue
            yield np.array(img.crop((x, y, x + w, y + h))), 1

        # negative samples (background)
        for _ in range(len(bboxes)):
            rx = random.randint(0, max(0, W - 64))
            ry = random.randint(0, max(0, H - 64))
            yield np.array(img.crop((rx, ry, rx + 64, ry + 64))), 0


def val_generator():
    for img_id in val_img_ids:
        img_name = img_id_to_name[img_id]
        img_path = os.path.join(TRAIN_IMG_DIR, img_name)
        if not os.path.exists(img_path):
            continue

        img = Image.open(img_path).convert("RGB")
        W, H = img.size
        bboxes = img_to_bboxes[img_id]

        # positive
        for x, y, w, h in bboxes:
            x, y, w, h = map(int, [x, y, w, h])
            if w * h < MIN_CRACK_AREA:
                continue
            yield np.array(img.crop((x, y, x + w, y + h))), 1

        # negative
        for _ in range(len(bboxes)):
            rx = random.randint(0, max(0, W - 64))
            ry = random.randint(0, max(0, H - 64))
            yield np.array(img.crop((rx, ry, rx + 64, ry + 64))), 0


In [ ]:
# Resize + normalize
def preprocess_tf(img, label):
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

train_ds = tf.data.Dataset.from_generator(
    train_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).map(preprocess_tf).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_generator(
    val_generator,
    output_signature=(
        tf.TensorSpec(shape=(None, None, 3), dtype=tf.uint8),
        tf.TensorSpec(shape=(), dtype=tf.int32)
    )
).map(preprocess_tf).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train / Val datasets ready for MobileViT")


Train / Val datasets ready for MobileViT


In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetV2S

# Load pretrained EfficientNetV2 backbone
base_model = EfficientNetV2S(
    include_top=False,
    weights="imagenet",
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # freeze backbone initially

# Classification head
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

hybrid_model = models.Model(inputs, outputs)

print("Hybrid model (EfficientNetV2) built successfully")


Hybrid model (EfficientNetV2) built successfully


In [ ]:
import tensorflow as tf

# Compile hybrid model
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Checkpoint: save best model based on validation loss
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/hybrid_checkpoint.keras",
    save_best_only=True,
    monitor="val_loss"
)

# Train model
history_hybrid = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint_cb]
)

# Save final trained model
hybrid_model.save("/content/drive/MyDrive/hybrid_model.keras")

print("Model-3 (Hybrid EfficientNetV2) trained and saved")


Epoch 1/20
    410/Unknown 298s 545ms/step - accuracy: 0.5995 - loss: 0.6452

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


410/410 ━━━━━━━━━━━━━━━━━━━━ 360s 698ms/step - accuracy: 0.5996 - loss: 0.6451 - val_accuracy: 0.6582 - val_loss: 0.6033
Epoch 2/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 30s 69ms/step - accuracy: 0.6541 - loss: 0.6043 - val_accuracy: 0.6616 - val_loss: 0.5986
Epoch 3/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 29s 68ms/step - accuracy: 0.6600 - loss: 0.5941 - val_accuracy: 0.6722 - val_loss: 0.5878
Epoch 4/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 28s 65ms/step - accuracy: 0.6614 - loss: 0.5865 - val_accuracy: 0.6609 - val_loss: 0.5906
Epoch 5/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.6571 - loss: 0.5856 - val_accuracy: 0.6695 - val_loss: 0.5842
Epoch 6/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 31s 71ms/step - accuracy: 0.6618 - loss: 0.5776 - val_accuracy: 0.6695 - val_loss: 0.5829
Epoch 7/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 31s 72ms/step - accuracy: 0.6742 - loss: 0.5730 - val_accuracy: 0.6749 - val_loss: 0.5783
Epoch 8/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 29s 67ms/step - accuracy: 0.6902 - loss: 0.5617 - val_accur

In [ ]:
# Unfreeze only top layers for fine-tuning
base_model.trainable = True

# Freeze most layers, fine-tune last ~20 layers
for layer in base_model.layers[:-20]:
    layer.trainable = False

print("Fine-tuning enabled for last layers")


Fine-tuning enabled for last layers


In [ ]:
hybrid_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# Checkpoint for fine-tuned model
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "/content/drive/MyDrive/hybrid_finetuned_checkpoint.keras",
    save_best_only=True,
    monitor="val_loss"
)

# Fine-tuning training
history_finetune = hybrid_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[checkpoint_cb]
)

# Save final fine-tuned model
hybrid_model.save("/content/drive/MyDrive/hybrid_finetuned_model.keras")

print("Hybrid model fine-tuned and saved")


Epoch 1/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 121s 164ms/step - accuracy: 0.6451 - loss: 0.5873 - val_accuracy: 0.6629 - val_loss: 0.5720
Epoch 2/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 30s 68ms/step - accuracy: 0.6696 - loss: 0.5669 - val_accuracy: 0.6709 - val_loss: 0.5726
Epoch 3/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 31s 72ms/step - accuracy: 0.6756 - loss: 0.5638 - val_accuracy: 0.5625 - val_loss: 0.5719
Epoch 4/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 41s 98ms/step - accuracy: 0.6582 - loss: 0.5761 - val_accuracy: 0.6715 - val_loss: 0.5675
Epoch 5/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 41s 96ms/step - accuracy: 0.6910 - loss: 0.5549 - val_accuracy: 0.6762 - val_loss: 0.5663
Epoch 6/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 70s 68ms/step - accuracy: 0.6781 - loss: 0.5691 - val_accuracy: 0.6682 - val_loss: 0.5692
Epoch 7/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 31s 73ms/step - accuracy: 0.6790 - loss: 0.5612 - val_accuracy: 0.6848 - val_loss: 0.5522
Epoch 8/20
410/410 ━━━━━━━━━━━━━━━━━━━━ 31s 72ms/step - accuracy: 0.6874 - loss: 0.5498 